# Linear regression GPU baseline (PyTorch lstsq)

**Doc:** `torch.linalg.lstsq` on `[StandardScaler(X.fillna(0)) | 1]` is **exact OLS with intercept** on the scaled design — same as `sklearn` `Pipeline(StandardScaler, LinearRegression)` for that preprocessing. CPU: pandas, PCA, `GroupKFold`. GPU: tensor solve per fold.

Load `full_dataset_gpu.csv` / `genotypes_*_gpu.csv` when present. Saves `lr_gpu_weights.npy`, `lr_gpu_oof.csv`.

In [1]:
# Cell 1 — imports & load
import json
import subprocess
import sys
import time
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import yaml
from IPython.display import display
from scipy import stats
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid", context="notebook")
sns.set_style("whitegrid")

_here = Path.cwd().resolve()
ROOT = _here if (_here / "maize_gxe_ml_pipeline.py").exists() else _here.parent
if not (ROOT / "maize_gxe_ml_pipeline.py").exists() and _here.name.lower() == "output":
    ROOT = _here.parent
sys.path.insert(0, str(ROOT))

from maize_gxe_ml_pipeline import (
    load_config,
    metrics_dict,
)

CFG_PATH = ROOT / "pipeline_config.yaml"
cfg = load_config(CFG_PATH)
out = Path(cfg["output_dir"])
if not out.is_absolute():
    out = (CFG_PATH.parent / out).resolve()

SUF = str(cfg.get("artifact_suffix", "_gpu"))
cohort = str(cfg["cohort"]).upper()
tr_min, tr_max = int(cfg["years"]["train_min"]), int(cfg["years"]["train_max"])
te_year = int(cfg["years"]["test_year"])
pca_n = int(cfg["hypers"]["pca_components"])
n_splits = int(cfg["hypers"]["group_cv_splits"])
rs = int(cfg.get("random_state", 42))
rng = np.random.default_rng(rs)

full_path = out / f"full_dataset{SUF}.csv"
if not full_path.exists():
    full_path = out / "full_dataset.csv"
geno_path = out / f"genotypes_{cohort.lower()}{SUF}.csv"
if not geno_path.exists():
    geno_path = out / f"genotypes_{cohort.lower()}.csv"

assert full_path.exists(), f"Missing {full_path} — run pipeline first."
assert geno_path.exists(), f"Missing {geno_path} — run pipeline first."

full_dataset = pd.read_csv(full_path, low_memory=False)
geno = pd.read_csv(geno_path, low_memory=False)
snp_cols = [c for c in geno.columns if c != "line_id"]

print("torch.cuda.is_available():", torch.cuda.is_available())
print(out, full_path.name, full_dataset.shape, geno_path.name, geno.shape)

torch.cuda.is_available(): True
C:\Users\kensm\Gods-Mercy\output full_dataset.csv (530986, 124) genotypes_c1.csv (76705, 2501)


In [2]:
# Cell 2 — CPU scale + NaN handle; rebuild modeling frame (same as CPU notebook)
from maize_gxe_ml_pipeline import (
    fit_pca_for_lines,
    map_pca_to_rows,
    step5_environment_features,
    step6_select_env_and_gxe,
)

train_year_mask = full_dataset["YEAR"].between(tr_min, tr_max) & full_dataset["YLDBE"].notna()
test_year_mask = (full_dataset["YEAR"] == te_year) & full_dataset["YLDBE"].notna()
model_df = full_dataset.loc[train_year_mask | test_year_mask].copy()
model_df = model_df[model_df["line_id"].isin(set(geno["line_id"]))].copy()

max_rows = cfg.get("max_train_rows")
per_env = cfg.get("max_rows_per_env")
train_only = model_df.loc[model_df["YEAR"].between(tr_min, tr_max)]
if per_env:
    parts = []
    for env_id, chunk in train_only.groupby("env_id"):
        if len(chunk) > int(per_env):
            parts.append(chunk.sample(n=int(per_env), random_state=int(rng.integers(0, 1_000_000))))
        else:
            parts.append(chunk)
    train_only = pd.concat(parts, axis=0)
if max_rows is not None and len(train_only) > int(max_rows):
    train_only = train_only.sample(n=int(max_rows), random_state=42)
test_only = model_df.loc[model_df["YEAR"] == te_year]
model_df = pd.concat([train_only, test_only], axis=0).drop_duplicates()
snp_in_m = [c for c in snp_cols if c in model_df.columns]
if snp_in_m:
    model_df = model_df.drop(columns=snp_in_m, errors="ignore")

model_df, env_numeric, _ = step5_environment_features(model_df, str(cfg.get("env_scaling", "global")))
train_lines = model_df.loc[model_df["YEAR"].between(tr_min, tr_max), "line_id"].unique()
_, final_pca_pipe = fit_pca_for_lines(geno, snp_cols, train_lines, pca_n, rs)
gpc_df = map_pca_to_rows(geno, snp_cols, model_df["line_id"], final_pca_pipe)
gpc_cols = list(gpc_df.columns)
for c in gpc_cols:
    model_df[c] = gpc_df[c].values

is_train = model_df["YEAR"].between(tr_min, tr_max).values
env_only_numeric = [c for c in env_numeric if c in model_df.columns and not str(c).startswith("GPC_")]
engxe_path = out / f"env_gxe_feature_columns{SUF}.csv"
if not engxe_path.exists():
    engxe_path = out / "env_gxe_feature_columns.csv"
if engxe_path.exists():
    nongeno_feature_cols = pd.read_csv(engxe_path)["env_and_gxe_column"].astype(str).tolist()
    for c in nongeno_feature_cols:
        if c in model_df.columns:
            continue
        if c.startswith("GXE__") and "__x__" in c:
            rest = c[len("GXE__") :]
            gc, ec = rest.split("__x__", 1)
            model_df[c] = model_df[gc].astype(float) * model_df[ec].astype(float)
        else:
            raise KeyError(c)
else:
    past_cols = [c for c in ("past_yld_mean", "past_yld_median") if c in model_df.columns]
    env_chosen, pair_list = step6_select_env_and_gxe(
        model_df.loc[is_train].copy(), env_only_numeric, gpc_cols, rng,
        int(cfg.get("max_env_features_gxe", 20)), int(cfg.get("n_interaction_pairs", 10)),
    )
    interact_cols = []
    for gc, ec in pair_list:
        cname = f"GXE__{gc}__x__{ec}"
        model_df[cname] = model_df[gc].astype(float) * model_df[ec].astype(float)
        interact_cols.append(cname)
    nongeno_feature_cols = [c for c in env_chosen + past_cols + interact_cols if c in model_df.columns]

feature_cols = [c for c in gpc_cols + nongeno_feature_cols if c in model_df.columns]
model_df = model_df.reset_index(drop=True)
is_train = model_df["YEAR"].between(tr_min, tr_max).values
G = model_df[gpc_cols].apply(pd.to_numeric, errors="coerce").fillna(0.0).values
Ng = model_df[nongeno_feature_cols].apply(pd.to_numeric, errors="coerce").fillna(0.0).values
X_all = np.hstack([G, Ng])
y_all = model_df["YLDBE"].values.astype(float)
groups_all = model_df["env_id"].astype(str).values
train_idx = is_train
print("features:", len(feature_cols), "rows:", len(model_df))

C:\Users\kensm\AppData\Local\Temp\ipykernel_32500\3491954277.py:39: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  model_df[c] = gpc_df[c].values
C:\Users\kensm\AppData\Local\Temp\ipykernel_32500\3491954277.py:39: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  model_df[c] = gpc_df[c].values
C:\Users\kensm\AppData\Local\Temp\ipykernel_32500\3491954277.py:39: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns 

features: 82 rows: 82522


C:\Users\kensm\AppData\Local\Temp\ipykernel_32500\3491954277.py:54: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  model_df[c] = model_df[gc].astype(float) * model_df[ec].astype(float)
C:\Users\kensm\AppData\Local\Temp\ipykernel_32500\3491954277.py:54: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  model_df[c] = model_df[gc].astype(float) * model_df[ec].astype(float)
C:\Users\kensm\AppData\Local\Temp\ipykernel_32500\3491954277.py:54: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.

In [3]:
# Cell 3 — GPU linear solve: GroupKFold OOF + final lstsq; nvidia-smi + timeit
def check_gpu_mem_mib():
    try:
        r = subprocess.run(
            ["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
            capture_output=True, text=True, timeout=15, check=False,
        )
        if r.returncode != 0 or not (r.stdout or "").strip():
            return None
        return float((r.stdout or "").strip().splitlines()[0])
    except (FileNotFoundError, subprocess.TimeoutExpired, ValueError, IndexError):
        return None

def ols_lstsq_gpu(X_np, y_np, device):
    X_np = np.asarray(X_np, dtype=np.float64)
    y_np = np.asarray(y_np, dtype=np.float64).reshape(-1, 1)
    n = X_np.shape[0]
    X_t = torch.tensor(X_np, device=device, dtype=torch.float32)
    y_t = torch.tensor(y_np, device=device, dtype=torch.float32)
    ones = torch.ones((n, 1), device=device, dtype=torch.float32)
    Xa = torch.cat([X_t, ones], dim=1)
    sol = torch.linalg.lstsq(Xa, y_t).solution
    w = sol[:-1, 0].detach().cpu().numpy()
    b = float(sol[-1, 0].detach().cpu().item())
    return w, b

def predict_gpu(X_np, w, b, device):
    with torch.no_grad():
        Xt = torch.tensor(np.asarray(X_np, dtype=np.float32), device=device, dtype=torch.float32)
        wt = torch.tensor(w.astype(np.float32), device=device, dtype=torch.float32)
        return (Xt @ wt.reshape(-1) + b).cpu().numpy()

device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cpu":
    print("GPU unavailable; falling back to CPU sklearn-style lstsq (same math).")

df_train = model_df.loc[train_idx].reset_index(drop=True)
y_tr = y_all[train_idx]
g_tr = groups_all[train_idx]
gkf = GroupKFold(n_splits=n_splits)
oof_train = np.full(len(y_tr), np.nan)
fold_metrics = []

print("nvidia-smi pre-OOF:", check_gpu_mem_mib())
t0 = time.perf_counter()
for tr, te in gkf.split(df_train, y_tr, groups=g_tr):
    tr_lines = df_train.iloc[tr]["line_id"].unique()
    _, pipe = fit_pca_for_lines(geno, snp_cols, tr_lines, pca_n, rs)
    gpc_tr = map_pca_to_rows(geno, snp_cols, df_train.iloc[tr]["line_id"], pipe).values
    gpc_te = map_pca_to_rows(geno, snp_cols, df_train.iloc[te]["line_id"], pipe).values
    Xtr = np.nan_to_num(np.hstack([gpc_tr, df_train.iloc[tr][nongeno_feature_cols].values.astype(float)]), nan=0.0)
    Xte = np.nan_to_num(np.hstack([gpc_te, df_train.iloc[te][nongeno_feature_cols].values.astype(float)]), nan=0.0)
    scaler = StandardScaler()
    Xtr_s = scaler.fit_transform(Xtr)
    Xte_s = scaler.transform(Xte)
    ytr = np.nan_to_num(y_tr[tr], nan=0.0)
    w, b = ols_lstsq_gpu(Xtr_s, ytr, device)
    oof_train[te] = predict_gpu(Xte_s, w, b, device)
    fold_metrics.append(metrics_dict(y_tr[te], oof_train[te]))
t_oof = time.perf_counter() - t0
print("nvidia-smi post-OOF:", check_gpu_mem_mib(), "| OOF wall:", round(t_oof, 3), "s")

oof_full = np.full(len(y_all), np.nan)
oof_full[np.where(train_idx)[0]] = oof_train
glob_oof = metrics_dict(y_tr, oof_train)
fold_rmses = [float(fm["rmse"]) for fm in fold_metrics if np.isfinite(fm["rmse"])]
cv_rmse_std = float(np.std(fold_rmses)) if fold_rmses else float("nan")

X_tr_all = np.nan_to_num(X_all[train_idx], nan=0.0)
y_tr_all = np.nan_to_num(y_all[train_idx], nan=0.0)
scaler_final = StandardScaler()
X_tr_s = scaler_final.fit_transform(X_tr_all)
t1 = time.perf_counter()
w_final, b_final = ols_lstsq_gpu(X_tr_s, y_tr_all, device)
t_fit = time.perf_counter() - t1
print("Final lstsq:", round(t_fit, 4), "s | OOF metrics:", glob_oof)

X_all_s = scaler_final.transform(np.nan_to_num(X_all, nan=0.0))
pred_all = predict_gpu(X_all_s, w_final, b_final, device)

np.save(out / "lr_gpu_weights.npy", np.concatenate([w_final.astype(np.float64), np.array([b_final])]))
joblib.dump(scaler_final, out / "lr_gpu_scaler.pkl")
lr_gpu_oof = model_df[["line_id", "env_id", "YEAR", "LOC", "YLDBE"]].copy()
lr_gpu_oof["oof_pred_lr_gpu"] = oof_full
lr_gpu_oof["pred_final_lr_gpu"] = pred_all
lr_gpu_oof.to_csv(out / "lr_gpu_oof.csv", index=False)

nvidia-smi pre-OOF: 307.0
nvidia-smi post-OOF: 510.0 | OOF wall: 1527.508 s
Final lstsq: 0.0227 s | OOF metrics: {'rmse': 5.242552447491263e-05, 'mae': 4.374183593745771e-05, 'r2': 0.9999999999979207}


### 4. Model parameters (interpretable OLS on scaled design)

**GPU note:** Coefficients from `torch.linalg.lstsq` match **CPU `sklearn` LinearRegression** on the same `(StandardScaler, X)` — these visuals are identical.

**Reading coefs:** On the **standardized** design, a positive coefficient means **+1 SD** in that feature associates with **+coef** bushels/acre, holding others fixed (linear assumption). The intercept is **predicted yield when all scaled features are 0** (i.e., at each feature's training mean).

*Linear assumes independent additive linear effects; residual plots (below) test whether that is plausible for maize GxE.*

In [4]:
# Cell 4 — model params (post-OOF fit): intercept, coef_df, bucket sums
X_cols = feature_cols
intercept = float(b_final)

coef_df = pd.DataFrame({"Feature": X_cols, "Coef": w_final})
coef_df["abs_coef"] = coef_df["Coef"].abs()
coef_df = coef_df.sort_values("abs_coef", ascending=False).drop(columns="abs_coef")


def bucket(name: str) -> str:
    if name.startswith("GPC_"):
        return "Genetic"
    if name.startswith("GXE__"):
        return "GxE"
    if any(x in name.upper() for x in ("PRCP", "TAVG", "HTDD", "CLDD", "DP01", "DP10")):
        return "Weather"
    if any(x in name.upper() for x in ("CLAY", "SILT", "SAND", "PH", "SOC", "NITROGEN", "CFVO")):
        return "Soil"
    if "past_yld" in name:
        return "Past_pheno"
    return "Other"


coef_df["bucket"] = coef_df["Feature"].map(bucket)
bucket_sums = coef_df.groupby("bucket")["Coef"].sum().to_dict()

tr_df = model_df.loc[train_idx].copy()
y = tr_df["YLDBE"].values
oof_preds = oof_train

print(f"Intercept (baseline yield when scaled feats = 0 / at feature means): {intercept:.2f} bu/ac")
print("\nTop 10 Coefs (Δ yield per +1 SD of scaled feature, others fixed):")
print(coef_df.drop(columns="bucket").head(10).round(4))
print("\nBucket impacts (sum of coefs in scaled space):")
print(pd.Series(bucket_sums).sort_values(ascending=False).round(2))
print("\nPositive coef: +1 SD of that feature → +coef bu/ac (linear, holding others fixed).")

Intercept (baseline yield when scaled feats = 0 / at feature means): 192.27 bu/ac

Top 10 Coefs (Δ yield per +1 SD of scaled feature, others fixed):
          Feature       Coef
65         YLD_BE  36.356701
64  phh2o_15_30cm  -0.000000
41         GPC_42  -0.000000
4           GPC_5   0.000000
13         GPC_14   0.000000
57       X10_CLDD   0.000000
70  past_yld_mean   0.000000
58       X04_CLDD   0.000000
55       X09_CLDD  -0.000000
56       X10_HTDD  -0.000000

Bucket impacts (sum of coefs in scaled space):
Other         36.36
Genetic        0.00
Past_pheno     0.00
Weather        0.00
GxE           -0.00
Soil          -0.00
dtype: float64

Positive coef: +1 SD of that feature → +coef bu/ac (linear, holding others fixed).


### 5. Actual vs predicted (OOF)

**Perfect fit** = points on the diagonal. **Spread** = prediction error. Gaps vs the diagonal motivate **non-linear** models (e.g. XGBoost GPU) when GxE is curved or interaction-heavy.

In [5]:
# Cell 5 — actual vs predicted (OOF train rows)
plt.figure(figsize=(8, 8))
plt.scatter(y, oof_preds, alpha=0.6, s=10)
plt.plot([y.min(), y.max()], [y.min(), y.max()], "r--", lw=2)
plt.xlabel("Actual YLDBE (bu/ac)")
plt.ylabel("Predicted (OOF)")
plt.title("OOF: Actual vs Predicted Yield")
plt.savefig(out / "lr_actual_pred.png", dpi=150, bbox_inches="tight")
plt.show()

C:\Users\kensm\AppData\Local\Temp\ipykernel_32500\3592396249.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 6. Residual diagnostics

**Residuals vs predicted:** random cloud is consistent with **homoscedastic** linear errors; funnels suggest **misspecification**. **Residuals by LOC** flag **env bias** (GxE). **Histogram** and **top coef** panel complement the global fit table.

In [6]:
# Cell 6 — residuals: vs pred, by LOC, top coefs, histogram
residuals = y - oof_preds
full_df = pd.DataFrame(
    {
        "Pred": oof_preds,
        "Resid": residuals,
        "LOC": tr_df["LOC"].astype(str).values,
        "YEAR": tr_df["YEAR"].values,
        "Line": tr_df["line_id"].values,
    }
)

fig, axes = plt.subplots(2, 2, figsize=(15, 12))
axes[0, 0].scatter(full_df["Pred"], full_df["Resid"], alpha=0.6, s=10)
axes[0, 0].axhline(0, color="r", ls="--", lw=2)
axes[0, 0].set_xlabel("Predicted")
axes[0, 0].set_ylabel("Residual")
axes[0, 0].set_title("Residuals vs Predicted (random scatter = good linear fit)")

if full_df["LOC"].nunique() > 30:
    top_locs = full_df["LOC"].value_counts().head(30).index
    sub_loc = full_df[full_df["LOC"].isin(top_locs)]
else:
    sub_loc = full_df
sns.boxplot(data=sub_loc, x="LOC", y="Resid", ax=axes[0, 1])
axes[0, 1].axhline(0, color="r", ls="--", lw=2)
axes[0, 1].set_xticklabels(axes[0, 1].get_xticklabels(), rotation=45, ha="right")
axes[0, 1].set_title("Residuals by LOC (pattern → env bias / GxE)")

top20_coef = coef_df.head(20).iloc[::-1]
sns.barplot(data=top20_coef, y="Feature", x="Coef", hue="bucket", dodge=False, ax=axes[1, 0])
axes[1, 0].axvline(0, color="gray", lw=0.8)
axes[1, 0].set_title("Top feature coefficients (absolute)")

sns.histplot(residuals, kde=True, ax=axes[1, 1])
axes[1, 1].axvline(0, color="r", ls="--", lw=2)
axes[1, 1].set_title("Residual distribution (normality check)")

plt.tight_layout()
plt.savefig(out / "lr_diagnostics.png", dpi=150, bbox_inches="tight")
plt.show()

C:\Users\kensm\AppData\Local\Temp\ipykernel_32500\1351299508.py:27: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  axes[0, 1].set_xticklabels(axes[0, 1].get_xticklabels(), rotation=45, ha="right")
C:\Users\kensm\AppData\Local\Temp\ipykernel_32500\1351299508.py:41: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 7. Maize-specific partial views & summary

**Pred vs key traits:** exploratory **partial** relationships (linear model combines all features; these are **bivariate** checks). **Q-Q plot:** if residuals deviate from the diagonal at tails, consider **YLDBE transform** or **non-linear** / **tree** models.

**Takeaway:** Residuals patterned by **LOC** → strengthen **GxE** / env features. **Non-normal** residuals → consider transforms or **XGBoost GPU** baseline comparison.

In [7]:
# Cell 7 — maize scatterplots, Q-Q, summary table, CSV/JSON


def _pick_col(candidates):
    for c in candidates:
        if c in model_df.columns:
            return c
    return None


key_feats = [
    c
    for c in [
        _pick_col(["PHT", "PHT_BE", "pht"]),
        _pick_col(["past_yld_mean", "past_yld_median"]),
        _pick_col(["summer_tavg"]),
    ]
    if c is not None
]

plot_df = tr_df.copy()
plot_df["Pred"] = oof_preds
for feat in key_feats:
    plt.figure(figsize=(8, 6))
    sns.scatterplot(data=plot_df, x=feat, y="Pred", alpha=0.6, s=12)
    plt.xlabel(feat)
    plt.ylabel("Predicted YLDBE (OOF)")
    plt.title(f"Pred yield vs {feat}")
    safe = str(feat).replace("/", "_").replace(" ", "_")
    plt.savefig(out / f"lr_vs_{safe}.png", dpi=150, bbox_inches="tight")
    plt.show()

fig, ax = plt.subplots(figsize=(8, 6))
stats.probplot(residuals, dist="norm", plot=ax)
ax.set_title("Q-Q plot: residuals vs normal")
plt.savefig(out / "lr_qq.png", dpi=150, bbox_inches="tight")
plt.show()

summary = pd.DataFrame(
    {
        "Metric": ["R2 (OOF)", "RMSE (bu/ac)", "MAE", "Resid mean", "Resid std"],
        "Value": [
            r2_score(y, oof_preds),
            np.sqrt(mean_squared_error(y, oof_preds)),
            mean_absolute_error(y, oof_preds),
            float(np.mean(residuals)),
            float(np.std(residuals)),
        ],
    }
).round(3)
print("LinearRegression (GPU lstsq) summary:\n", summary.to_string(index=False))
summary.to_json(out / "lr_visual_summary.json", orient="records", indent=2)

coef_df.drop(columns="bucket", errors="ignore").to_csv(out / "lr_coefs.csv", index=False)
full_df.to_csv(out / "lr_interpretability_df.csv", index=False)

print("LR visuals added – interpretable baseline ready for hack demo!")

C:\Users\kensm\AppData\Local\Temp\ipykernel_32500\281873534.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\kensm\AppData\Local\Temp\ipykernel_32500\281873534.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


LinearRegression (GPU lstsq) summary:
       Metric  Value
    R2 (OOF)    1.0
RMSE (bu/ac)    0.0
         MAE    0.0
  Resid mean   -0.0
   Resid std    0.0
LR visuals added – interpretable baseline ready for hack demo!


C:\Users\kensm\AppData\Local\Temp\ipykernel_32500\281873534.py:37: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Yield Analysis by US Region

Group trials by agricultural regions using `LATITUDE`/`LONGITUDE` and `LOC` codes, then compare **actual vs predicted** average `YLDBE` (bu/ac).

This supports the hackathon narrative: **Corn Belt environments often show higher yields; regional residual patterns reveal GxE structure.**

In [11]:
# Cell 8 — Yield by US Region (geo EDA)
# Optional geo libs (fallbacks are built in)
try:
    import geopandas as gpd  # noqa: F401
except Exception:
    gpd = None

try:
    import plotly.express as px
except Exception:
    px = None

from sklearn.cluster import KMeans

# Build a robust working frame from existing OOF context
if "full_df" in globals():
    region_df = full_df.copy()
else:
    region_df = pd.DataFrame(
        {
            "LOC": tr_df["LOC"].astype(str).values,
            "YEAR": tr_df["YEAR"].values,
            "YLDBE": tr_df["YLDBE"].values,
            "Pred": oof_preds,
        }
    )

if "YLDBE" not in region_df.columns:
    region_df["YLDBE"] = tr_df["YLDBE"].values
if "Pred" not in region_df.columns:
    region_df["Pred"] = oof_preds
if "LOC" not in region_df.columns:
    region_df["LOC"] = tr_df["LOC"].astype(str).values
if "YEAR" not in region_df.columns:
    region_df["YEAR"] = tr_df["YEAR"].values

# Ensure geocoordinates exist when available in model_df/tr_df
for c in ["LONGITUDE", "LATITUDE"]:
    if c not in region_df.columns:
        if c in tr_df.columns:
            region_df[c] = tr_df[c].values
        elif c in model_df.columns:
            # align by LOC+YEAR median location
            loc_year_geo = model_df.groupby(["LOC", "YEAR"], as_index=False)[c].median()
            region_df = region_df.merge(loc_year_geo, on=["LOC", "YEAR"], how="left")
        else:
            region_df[c] = np.nan

# 1) Region mapping by known site codes + fallback by state prefix
region_map = {
    "Corn Belt": ["IALI", "IAFO", "IASP", "IADA", "IAPR", "ILMN", "INVI"],
    "Northern Plains": ["NEDA", "MNHE", "MNOW", "SD"],
    "Central Plains": ["MOBU", "KS", "NE"],
    "South": ["TX", "OK", "AR", "LA", "MS", "AL", "GA", "FL"],
}


def infer_region(loc):
    s = str(loc).upper()
    for region, codes in region_map.items():
        if s in codes:
            return region
        if any(s.startswith(code) for code in codes):
            return region
    if s.startswith(("IA", "IL", "IN", "OH", "MO")):
        return "Corn Belt"
    if s.startswith(("MN", "ND", "SD")):
        return "Northern Plains"
    if s.startswith(("NE", "KS")):
        return "Central Plains"
    if s.startswith(("TX", "OK", "AR", "LA", "MS", "AL", "GA", "FL")):
        return "South"
    return "Other"


region_df["Region"] = region_df["LOC"].map(infer_region)

# Fallback: KMeans geo clusters when many sites remain uncategorized
other_frac = float((region_df["Region"] == "Other").mean())
if other_frac > 0.30 and region_df[["LONGITUDE", "LATITUDE"]].notna().all(axis=1).sum() >= 20:
    coords = region_df[["LONGITUDE", "LATITUDE"]].copy()
    coords = coords.fillna(coords.median(numeric_only=True)).fillna(0.0)
    km = KMeans(n_clusters=4, random_state=42, n_init=10)
    labels = km.fit_predict(coords.values)
    region_df["Region"] = [f"Region {chr(65 + i)}" for i in labels]

# 2) Summary tables
region_summary = (
    region_df.groupby(["Region", "YEAR"], as_index=False)
    .agg(
        Mean_Yield=("YLDBE", "mean"),
        Std_Yield=("YLDBE", "std"),
        N_Trials=("YLDBE", "count"),
        Mean_Pred=("Pred", "mean"),
    )
    .round(1)
)
print("Yield by Region/Year (bu/ac):")
display(region_summary)
region_summary.to_csv(out / "yield_by_region_year.csv", index=False)

region_overall = (
    region_df.groupby("Region", as_index=True)
    .agg(
        Mean_Yield=("YLDBE", "mean"),
        Std_Yield=("YLDBE", "std"),
        N_Trials=("YLDBE", "count"),
        Mean_Pred=("Pred", "mean"),
    )
    .round(1)
    .sort_values("Mean_Yield", ascending=False)
)
print("\nOverall by Region:")
display(region_overall)

# 3) Static visuals
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
region_overall[["Mean_Yield", "Mean_Pred"]].plot(kind="bar", ax=axes[0], width=0.6)
axes[0].set_title("Avg Yield: Actual vs Predicted by Region")
axes[0].set_ylabel("Yield (bu/ac)")
axes[0].legend(["Actual", "Pred"])

sns.boxplot(data=region_df, x="Region", y="YLDBE", ax=axes[1])
axes[1].set_title("Yield Distribution by Region")
axes[1].tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.savefig(out / "yield_by_region.png", dpi=150, bbox_inches="tight")
plt.show()

# 4) Interactive geo map (if plotly available and coords present)
def _save_plotly_png(fig, path):
    try:
        fig.write_image(path, width=1200, height=800, scale=2)
        print("PNG map saved:", path)
    except Exception as e:
        print(f"PNG export skipped for {path.name}: {e} (install kaleido: pip install -U kaleido)")


if px is not None and region_df[["LATITUDE", "LONGITUDE"]].notna().all(axis=1).any():
    map_df = (
        region_df.dropna(subset=["LATITUDE", "LONGITUDE"])
        .groupby(["Region", "LOC"], as_index=False)
        .agg(
            LATITUDE=("LATITUDE", "mean"),
            LONGITUDE=("LONGITUDE", "mean"),
            Mean_Yield=("YLDBE", "mean"),
            Mean_Pred=("Pred", "mean"),
            N_Trials=("YLDBE", "count"),
        )
        .round(2)
    )
    size_col = "Mean_Yield"
    fig_map = px.scatter_geo(
        map_df,
        lat="LATITUDE",
        lon="LONGITUDE",
        color="Region",
        size=size_col,
        hover_data=["LOC", "Mean_Yield", "Mean_Pred", "N_Trials"],
        title="Trial Yields by US Region (Bubble = Avg Yield)",
        scope="usa",
    )
    fig_map.write_html(out / "yield_map.html")
    _save_plotly_png(fig_map, out / "yield_map.png")
    print("Map saved:", out / "yield_map.html")
else:
    print("Plotly map skipped (plotly missing or no valid LAT/LONG).")

print("Insight: Corn Belt should trend higher mean yields; regional spread indicates GxE complexity.")
print("Yield by US Region added – geo EDA for demo slides")

Yield by Region/Year (bu/ac):


,Region,YEAR,Mean_Yield,Std_Yield,N_Trials,Mean_Pred
0,Central Plains,2001,196.1,28.3,924,196.1
1,Central Plains,2002,194.2,35.8,429,194.2
2,Central Plains,2003,214.0,29.0,585,214.0
3,Central Plains,2004,222.8,27.0,1231,222.8
4,Central Plains,2005,214.4,33.1,964,214.4
5,Central Plains,2006,187.8,47.3,905,187.8
6,Central Plains,2007,205.7,31.7,620,205.7
7,Corn Belt,2001,181.4,36.1,2655,181.4
8,Corn Belt,2002,181.5,38.2,2633,181.5
9,Corn Belt,2003,178.0,35.4,3925,178.0



Overall by Region:


,Mean_Yield,Std_Yield,N_Trials,Mean_Pred
Region,,,,
Central Plains,206.5,35.9,5658,206.5
Corn Belt,192.7,34.9,24917,192.7
Northern Plains,189.3,34.2,6310,189.3
Other,183.6,39.7,6403,183.6
South,182.5,39.2,1712,182.5


Map saved: C:\Users\kensm\Gods-Mercy\output\yield_map.html
Insight: Corn Belt should trend higher mean yields; regional spread indicates GxE complexity.
Yield by US Region added – geo EDA for demo slides


C:\Users\kensm\AppData\Local\Temp\ipykernel_32500\1333125063.py:128: UserWarning:

FigureCanvasAgg is non-interactive, and thus cannot be shown



In [8]:
# Cell 8 — metrics by env, GBR compare, residual/coef slides, rankings, lr_gpu_summary.json
display(pd.DataFrame([glob_oof]).T.rename(columns={0: "value"}))

by_env_rows = []
for eid, chunk in tr_df.groupby("env_id"):
    m = metrics_dict(chunk["YLDBE"].values, oof_full[chunk.index.values])
    m["env_id"] = eid
    m["n"] = len(chunk)
    by_env_rows.append(m)
metrics_by_env = pd.DataFrame(by_env_rows).sort_values("rmse", na_position="last")
metrics_by_env.to_csv(out / "lr_gpu_metrics_by_env_oof.csv", index=False)

summary_gpu = out / f"run_summary{SUF}.json"
if summary_gpu.exists():
    with open(summary_gpu, encoding="utf-8") as f:
        summ = json.load(f)
    go = summ.get("oof_global_metrics", {})
    display(
        pd.DataFrame(
            {
                "model": ["Tree pipeline OOF", "PyTorch lstsq OOF"],
                "rmse": [go.get("rmse"), glob_oof["rmse"]],
                "mae": [go.get("mae"), glob_oof["mae"]],
                "r2": [go.get("r2"), glob_oof["r2"]],
            }
        )
    )

fig, ax = plt.subplots(figsize=(7, 5))
ax.axhline(0, color="gray", lw=0.8)
ax.scatter(oof_train, tr_df["YLDBE"].values - oof_train, alpha=0.2, s=10, edgecolors="none")
ax.set_xlabel("OOF pred")
ax.set_ylabel("Residual")
ax.set_title("PyTorch lstsq OOF residuals")
fig.tight_layout()
fig.savefig(out / "lr_gpu_residuals.png", dpi=150)
plt.show()

feat_df = coef_df.rename(columns={"Feature": "feat", "Coef": "coef"})
feat_df["abs_coef"] = feat_df["coef"].abs()
top20 = feat_df.sort_values("abs_coef", ascending=False).head(20)
fig, ax = plt.subplots(figsize=(9, 7))
sns.barplot(data=top20, y="feat", x="coef", hue="bucket", dodge=False, ax=ax)
ax.axvline(0, color="gray", lw=0.8)
ax.set_title("Top 20 |coef| (GPU lstsq, scaled design)")
fig.tight_layout()
fig.savefig(out / "lr_gpu_coef_plot.png", dpi=150, bbox_inches="tight")
plt.show()
display(feat_df.groupby("bucket")["abs_coef"].sum().reset_index())

topk = int(cfg.get("topk", 10))
te_mask = model_df["YEAR"].values == te_year
rank_rows = []
for env_id, chunk in model_df.loc[te_mask].groupby("env_id"):
    order = np.argsort(-pred_all[chunk.index.values])
    top_idx = chunk.index.values[order[:topk]]
    sub = model_df.loc[top_idx].copy()
    sub["pred_final_lr_gpu"] = pred_all[top_idx]
    sub["rank"] = np.arange(1, len(sub) + 1)
    sub["uncertainty_oof_rmse"] = glob_oof["rmse"]
    rank_rows.append(sub)
lr_rank = pd.concat(rank_rows, axis=0) if rank_rows else pd.DataFrame()
lr_rank.to_csv(out / "lr_gpu_rankings_topk_per_env.csv", index=False)

interp_pngs = [
    str(out / "lr_actual_pred.png"),
    str(out / "lr_diagnostics.png"),
    str(out / "lr_qq.png"),
] + sorted(str(p) for p in out.glob("lr_vs_*.png"))

lr_summary = {
    "model": "PyTorch lstsq OLS (GPU if CUDA)",
    "device": device,
    "oof_global_metrics": glob_oof,
    "cv_rmse_std": cv_rmse_std,
    "wall_seconds": {"oof_loop": float(t_oof), "final_lstsq": float(t_fit)},
    "outputs": [str(out / "lr_gpu_weights.npy"), str(out / "lr_gpu_oof.csv")],
    "interpretability_pngs": interp_pngs,
    "interpretability_tables": [str(out / "lr_coefs.csv"), str(out / "lr_visual_summary.json")],
}


def _jd(o):
    return o.item() if isinstance(o, (np.floating, np.integer)) else str(o)


with open(out / "lr_gpu_summary.json", "w", encoding="utf-8") as f:
    json.dump(lr_summary, f, indent=2, default=_jd)

print("GPU pipelines ready – run nvidia-smi during fit to confirm.")

,value
rmse,0.000052
mae,0.000044
r2,1.000000


C:\Users\kensm\AppData\Local\Temp\ipykernel_32500\3351942513.py:37: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\kensm\AppData\Local\Temp\ipykernel_32500\3351942513.py:48: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,bucket,abs_coef
0,Genetic,0.000039
1,GxE,0.000006
2,Other,36.356712
3,Past_pheno,0.000003
4,Soil,0.000008
5,Weather,0.000012


GPU pipelines ready – run nvidia-smi during fit to confirm.
